# Assignment 2: P+7 (Oulipian language modelling)
**Angeline Duong · CART 498**

This notebook adapts the Oulipo **N+7** technique (replace each noun with the noun seven entries later in a dictionary) into **P+7**: the last word of each line of Wallace Stevens' *The Snow Man* is replaced with the word that **GPT-2 ranks 7th most likely** to come next. The same code also produces a **P+x** version for any value of x.

**How it works, in short:**
1. Split each line into its beginning, its last word, and any punctuation.
2. Give GPT-2 the text leading up to the last word.
3. Rank GPT-2's predictions for the next word from most to least likely.
4. Take the word at rank x and put it back into the line with the original punctuation.

**Design choices:**
- **Whole words only.** GPT-2's vocabulary includes punctuation, line breaks and word pieces (such as "ing"). Only tokens that begin with a space and contain only letters are counted, so "rank 7" means the 7th-ranked *word*.
- **Context.** By default GPT-2 sees the title and the *original* poem up to the current line (`mode="poem"`), so each replacement is independent of the others.
- **Reproducible.** The code always picks the word at a fixed rank and never samples randomly, so the same x always gives the same poem.

## Step 1: Install libraries
Installs Hugging Face `transformers` (which provides GPT-2) and `torch` (which runs the model).

In [2]:
!pip install -q transformers torch

## Step 2: Load GPT-2
Loads the GPT-2 tokenizer, which converts text into token IDs, and the GPT-2 language model. It uses a GPU if one is available, otherwise the CPU. GPT-2's vocabulary has 50,257 tokens.

In [3]:
import re
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
tok = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
model.eval()
print("Loaded GPT-2 on", device, "| vocab size:", len(tok))

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loaded GPT-2 on cpu | vocab size: 50257


## Step 3: The original poem
Stores *The Snow Man* by Wallace Stevens as a list of 15 lines. The title is kept separately so it can be used as context for GPT-2 and added as a header to the output files.

In [4]:
TITLE = "The Snow Man\nby Wallace Stevens"

POEM = """One must have a mind of winter
To regard the frost and the boughs
Of the pine-trees crusted with snow;
And have been cold a long time
To behold the junipers shagged with ice,
The spruces rough in the distant glitter
Of the January sun; and not to think
Of any misery in the sound of the wind,
In the sound of a few leaves,
Which is the sound of the land
Full of the same wind
That is blowing in the same bare place
For the listener, who listens in the snow,
And, nothing himself, beholds
Nothing that is not there and the nothing that is."""

lines = POEM.split("\n")
print(len(lines), "lines")

15 lines


## Step 4: Split each line
A regular expression splits each line into three parts:
- the **beginning** of the line, which is kept as it is
- the **last word**, which gets replaced
- any **punctuation** after it, which is added back after the new word

For example, `Of the pine-trees crusted with snow;` becomes `('Of the pine-trees crusted with', 'snow', ';')`. The printout confirms all 15 lines split correctly.

In [5]:
LINE_RE = re.compile(r"^(.*?)([A-Za-z][A-Za-z'\-]*)([^A-Za-z]*)$")

def split_line(line):
    """'Of the pine-trees crusted with snow;' -> ('Of the pine-trees crusted with', 'snow', ';')"""
    prefix, word, punct = LINE_RE.match(line.rstrip()).groups()
    return prefix.rstrip(), word, punct

for line in lines:
    print(split_line(line))

('One must have a mind of', 'winter', '')
('To regard the frost and the', 'boughs', '')
('Of the pine-trees crusted with', 'snow', ';')
('And have been cold a long', 'time', '')
('To behold the junipers shagged with', 'ice', ',')
('The spruces rough in the distant', 'glitter', '')
('Of the January sun; and not to', 'think', '')
('Of any misery in the sound of the', 'wind', ',')
('In the sound of a few', 'leaves', ',')
('Which is the sound of the', 'land', '')
('Full of the same', 'wind', '')
('That is blowing in the same bare', 'place', '')
('For the listener, who listens in the', 'snow', ',')
('And, nothing himself,', 'beholds', '')
('Nothing that is not there and the nothing that', 'is', '.')


## Step 5: Build the context GPT-2 reads
This controls how much text GPT-2 sees before guessing the replacement word:
- `"line"`: only the current line up to the last word
- `"poem"` (**used in this notebook**): the title plus the original poem up to this point, so GPT-2 keeps the poem's wintry theme and each line is transformed independently
- `"chain"`: the *already transformed* lines, so each replacement affects the ones after it

In [6]:
def build_context(lines, done_lines, i, prefix, mode="poem"):
    """
    mode="line"  -> only the current line so far
    mode="poem"  -> the ORIGINAL poem up to this line, plus the current line so far
    mode="chain" -> the ALREADY-CHANGED poem up to this line (the changes build on each other)
    """
    if mode == "line":
        return prefix
    previous = lines[:i] if mode == "poem" else done_lines[:i]
    return TITLE + "\n\n" + "\n".join(previous + [prefix])

## Step 6: Rank GPT-2's predictions
`ranked_words()` asks GPT-2 for a score for every possible next token, converts the scores into probabilities, and sorts them from most to least likely.

A filter (`is_word`) removes tokens that aren't whole words: punctuation, line breaks and word pieces without a leading space. Without it, the 7th-ranked token would often be a comma or `\n` instead of a word.

The test below shows GPT-2's top 10 whole-word guesses for "One must have a mind of ___", with their probabilities.

In [7]:
# Mark which tokens count as real words: a leading space followed by letters only.
# This skips punctuation, newlines and word pieces like "ing".
vocab = [tok.decode([i]) for i in range(len(tok))]
is_word = torch.tensor([
    s.startswith(" ") and s[1:].isalpha() and s[1:].isascii() for s in vocab
])

def ranked_words(context, n):
    """Return the top-n whole-word guesses as (word, probability), most likely first."""
    ids = tok.encode(context, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(ids).logits[0, -1]           # scores for the next token
    probs = torch.softmax(logits, dim=-1)
    ranked = torch.argsort(logits, descending=True).cpu()
    ranked = ranked[is_word[ranked]]               # keep only real words
    return [(vocab[t].strip(), probs[t].item()) for t in ranked[:n].tolist()]

# Test: GPT-2's top 10 guesses for the end of line 1
prefix, word, punct = split_line(lines[0])
for rank, (w, p) in enumerate(ranked_words(prefix, 10), start=1):
    print(f"{rank:>2}. {w:<12} {p:.4f}")

 1. their        0.3277
 2. its          0.1343
 3. his          0.1304
 4. your         0.0248
 5. a            0.0202
 6. our          0.0169
 7. her          0.0162
 8. this         0.0123
 9. my           0.0116
10. one          0.0109


## Step 7: The P+x function
`p_plus_x(lines, x)` applies the technique to every line:
1. Split the line (Step 4)
2. Build the context (Step 5)
3. Get GPT-2's ranked words (Step 6) and take the one at rank **x**. Python counts from 0, so rank x is at index `x - 1`.
4. Rebuild the line with the original punctuation

With `show=True`, it prints each original word, its replacement, and the replacement's probability.

In [8]:
def p_plus_x(lines, x, mode="poem", show=True):
    done = []
    for i, line in enumerate(lines):
        prefix, word, punct = split_line(line)
        context = build_context(lines, done, i, prefix, mode)
        new_word, prob = ranked_words(context, x)[x - 1]   # rank x = position x-1
        new_line = f"{prefix} {new_word}{punct}"
        done.append(new_line)
        if show:
            print(f"{word:>10} -> {new_word:<12} (p={prob:.5f})")
    return "\n".join(done)

## Step 8: Generate P+7
Runs the function with **x = 7** and saves the result as `P+7.txt`.

The replacements stay close to the poem's theme (*wonder, chill, leaves, winter, summer*) because rank 7 is still near GPT-2's most likely guesses. Some are grammatically odd but correct under the rule, such as "a long *long*" and "not to *the*".

In [9]:
p7 = p_plus_x(lines, 7)
print("\n" + p7)

with open("P+7.txt", "w") as f:
    f.write(TITLE + "\n(P+7 version)\n\n" + p7 + "\n")

    winter -> wonder       (p=0.01399)
    boughs -> chill        (p=0.01353)
      snow -> leaves       (p=0.01357)
      time -> long         (p=0.00727)
       ice -> cold         (p=0.01165)
   glitter -> winter       (p=0.01237)
     think -> the          (p=0.02259)
      wind -> sun          (p=0.00932)
    leaves -> snow         (p=0.00878)
      land -> summer       (p=0.01005)
      wind -> winter       (p=0.00804)
     place -> space        (p=0.01207)
      snow -> winter       (p=0.01529)
   beholds -> does         (p=0.02024)
        is -> exists       (p=0.00639)

One must have a mind of wonder
To regard the frost and the chill
Of the pine-trees crusted with leaves;
And have been cold a long long
To behold the junipers shagged with cold,
The spruces rough in the distant winter
Of the January sun; and not to the
Of any misery in the sound of the sun,
In the sound of a few snow,
Which is the sound of the summer
Full of the same winter
That is blowing in the same bare space

## Step 9: Explore values of x
Prints the poem for different values of x to compare how the output changes. Add more numbers to the list to explore further.

In [13]:
for x in [34]:
    print(f"\n===== P+{x} =====")
    print(p_plus_x(lines, x, show=False))


===== P+34 =====
One must have a mind of admiration
To regard the frost and the fog
Of the pine-trees crusted with life;
And have been cold a long spring
To behold the junipers shagged with straw,
The spruces rough in the distant shade
Of the January sun; and not to my
Of any misery in the sound of the m,
In the sound of a few ch,
Which is the sound of the w
Full of the same earth
That is blowing in the same bare leaf
For the listener, who listens in the rain,
And, nothing himself, listens
Nothing that is not there and the nothing that happens.


## Step 10: Generate P+x (x = 34) and download
**x = 34** was chosen as the most absurd and witty version. The replacements move away from the obvious wintry vocabulary into surprising territory: "cold a long *spring*" is a small contradiction, and "the nothing that *happens*" gives Stevens' final line a new philosophical twist.

At this rank some lines also break down into word fragments (*m*, *ch*, *w*). These are real GPT-2 tokens that pass the letters-only filter, and they show how the model's suggestions stop being real words further down the ranking.

The result is saved as `P+34.txt`, and both text files are downloaded.

In [19]:
X = 34   # <- change this to the x you liked best

px = p_plus_x(lines, X)
print("\n" + px)

filename = f"P+{X}.txt"
with open(filename, "w") as f:
    f.write(TITLE + f"\n(P+{X} version)\n\n" + px + "\n")

from google.colab import files
files.download("P+7.txt")
files.download(filename)

    winter -> admiration   (p=0.00195)
    boughs -> fog          (p=0.00317)
      snow -> life         (p=0.00264)
      time -> spring       (p=0.00051)
       ice -> straw        (p=0.00240)
   glitter -> shade        (p=0.00378)
     think -> my           (p=0.00427)
      wind -> m            (p=0.00371)
    leaves -> ch           (p=0.00348)
      land -> w            (p=0.00331)
      wind -> earth        (p=0.00337)
     place -> leaf         (p=0.00355)
      snow -> rain         (p=0.00343)
   beholds -> listens      (p=0.00453)
        is -> happens      (p=0.00128)

One must have a mind of admiration
To regard the frost and the fog
Of the pine-trees crusted with life;
And have been cold a long spring
To behold the junipers shagged with straw,
The spruces rough in the distant shade
Of the January sun; and not to my
Of any misery in the sound of the m,
In the sound of a few ch,
Which is the sound of the w
Full of the same earth
That is blowing in the same bare leaf
For the l

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>